## Preprocess Metadata CSV

### Import Library

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

### Set Up Repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Clone repo & setup path
!git clone https://github.com/sg-riley/chest-xray-multilabel-classification.git /content/chest-xray-multilabel-classification 2>/dev/null || git -C /content/chest-xray-multilabel-classification pull

import sys
sys.path.insert(0, '/content/chest-xray-multilabel-classification')

from config.config import Config

### Konfigurasi Path

In [ ]:
cfg = Config()

print("Konfigurasi:")
print(f"   CSV input  : {cfg.CSV_RAW_PATH}")
print(f"   CSV output : {cfg.CSV_CLEANED_PATH}")
print(f"   Labels     : {cfg.TARGET_LABELS}")

# Validasi file input
import os
if os.path.exists(cfg.CSV_RAW_PATH):
    print("\nFile CSV input ditemukan")
else:
    print("\nFile CSV input TIDAK ditemukan - cek path di config/config.py")

### Load & Eksplorasi Data Mentah

In [ ]:
# Load CSV mentah
df_raw = load_metadata(cfg.CSV_RAW_PATH)

print()
print("5 baris pertama:")
display(df_raw.head())

print("\nInfo kolom:")
display(df_raw.dtypes.to_frame('dtype'))

# Cek missing values
missing = df_raw.isnull().sum()
print("\nMissing values:")
if missing.sum() == 0:
    print("  Tidak ada missing values")
else:
    display(missing[missing > 0])

# Distribusi semua label di data mentah
print("\nDistribusi semua label di data mentah:")
from collections import Counter
label_counter = Counter()
for finding in df_raw['Finding Labels']:
    for label in finding.split('|'):
        label_counter[label.strip()] += 1

df_all_labels = pd.DataFrame(
    label_counter.most_common(),
    columns=['Label', 'Jumlah']
)
df_all_labels['Persen (%)'] = (
    df_all_labels['Jumlah'] / len(df_raw) * 100
).round(2)
display(df_all_labels)

### Cleaning Metadata

In [ ]:
df_cleaned = clean_metadata(
    df       = df_raw,
    target_labels = cfg.TARGET_LABELS,
    finding_col   = 'Finding Labels'
)

print()
display(df_cleaned.head())

### Visualisasi Distribusi Label

In [ ]:
label_summary = []
for label in cfg.TARGET_LABELS + ['No_Finding']:
    cnt = df_cleaned[label].sum()
    pct = cnt / len(df_cleaned) * 100
    label_summary.append({
        'Label'     : label,
        'Jumlah'    : int(cnt),
        'Persen (%)': round(pct, 2),
        'Tipe'      : 'Normal' if label == 'No_Finding' else 'Penyakit'
    })
df_summary = pd.DataFrame(label_summary).sort_values('Jumlah', ascending=False)

COLORS_DISEASE = '#E74C3C'
COLOR_NORMAL   = '#2ECC71'
COLOR_MULTI    = '#3498DB'

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Distribusi Label — Metadata Setelah Cleaning",
             fontsize=14, fontweight='bold')

# Plot 1: Bar chart per label
ax1 = axes[0]
colors = [COLOR_NORMAL if l == 'No_Finding' else COLORS_DISEASE
          for l in df_summary['Label']]
bars = ax1.barh(df_summary['Label'], df_summary['Jumlah'],
                color=colors, edgecolor='white', height=0.6)
ax1.set_xlabel('Jumlah Gambar')
ax1.set_title('Jumlah per Label', fontweight='bold')
ax1.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, df_summary['Jumlah']):
    ax1.text(bar.get_width() + 300, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=8)
from matplotlib.patches import Patch
ax1.legend(handles=[
    Patch(color=COLORS_DISEASE, label='Penyakit'),
    Patch(color=COLOR_NORMAL,   label='Normal')
], fontsize=9)

# Plot 2: Pie chart Normal vs Penyakit
ax2 = axes[1]
n_disease = (df_cleaned['No_Finding'] == 0).sum()
n_normal  = (df_cleaned['No_Finding'] == 1).sum()
ax2.pie([n_disease, n_normal],
        labels=[f'Penyakit\n{n_disease:,}', f'Normal\n{n_normal:,}'],
        colors=[COLORS_DISEASE, COLOR_NORMAL],
        autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 10})
ax2.set_title('Proporsi Normal vs Penyakit', fontweight='bold')

# Plot 3: Distribusi label_count (jumlah label per gambar)
ax3 = axes[2]
label_dist = df_cleaned['label_count'].value_counts().sort_index()
bars3 = ax3.bar(label_dist.index.astype(str), label_dist.values,
                color=COLOR_MULTI, edgecolor='white')
ax3.set_xlabel('Jumlah Label Aktif per Gambar')
ax3.set_ylabel('Jumlah Gambar')
ax3.set_title('Distribusi Multi-Label', fontweight='bold')
for bar, val in zip(bars3, label_dist.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{val:,}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/content/label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisasi selesai")

### Simpan Metadata Bersih

In [ ]:
FINAL_COLUMNS = [
    # Kolom asli NIH
    'Image Index', 'Finding Labels', 'Follow-up #',
    'Patient ID', 'Patient Age', 'Patient Sex', 'View Position',
    'OriginalImage[Width', 'Height]',
    'OriginalImagePixelSpacing[x', 'y]',
    # Kolom one-hot label
    *cfg.TARGET_LABELS,
    'No_Finding',
    # Kolom utilitas
    'label_count'
]

# Pastikan semua kolom ada
available_cols = [c for c in FINAL_COLUMNS if c in df_cleaned.columns]
df_final = df_cleaned[available_cols]

# Simpan ke Drive
os.makedirs(os.path.dirname(cfg.CSV_CLEANED_PATH), exist_ok=True)
df_final.to_csv(cfg.CSV_CLEANED_PATH, index=False)

print("=" * 60)
print("HASIL CLEANING TERSIMPAN")
print("=" * 60)
print(f"   Path   : {cfg.CSV_CLEANED_PATH}")
print(f"   Baris  : {len(df_final):,}")
print(f"   Kolom  : {len(df_final.columns)}")
print()
print("   Kolom yang disimpan:")
for col in df_final.columns:
    print(f"   - {col}")